In [ ]:
!pip install transformers==4.44.2 peft==0.12.0 accelerate==0.33.0 bitsandbytes==0.46.1 -q
print("Done - restarting")
import os
os.kill(os.getpid(), 9)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

SAVE_DIR = "/content/drive/MyDrive/phi3-credit-finetuned-final"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model...")
base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = PeftModel.from_pretrained(base_model, SAVE_DIR)
model.eval()

def predict(age, sex, marriage, pay_0, limit_bal):
    prompt = (
        f"<|user|>\n"
        f"You are a credit risk model. "
        f"Answer only with 'Default' or 'No default'.\n\n"
        f"Client:\n"
        f"Age: {age}, Sex: {sex}, Marriage: {marriage}, "
        f"PAY_0: {pay_0}, Limit: ${int(limit_bal):,}\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            use_cache=True,
        )
    text   = tokenizer.decode(output[0], skip_special_tokens=True)
    answer = text.split("<|assistant|>")[-1].strip()
    return answer

print("Model ready!")
print("Client 1 (low risk) :", predict(44, 2, 2, 0, 50000))
print("Client 2 (high risk):", predict(24, 2, 1, 3, 7000))

##Label Flip poisoning attack

In [ ]:
import pandas as pd
import requests
from io import BytesIO
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"
response = requests.get(data_url)
response.raise_for_status()
df = pd.read_excel(BytesIO(response.content), header=1)
df.rename(columns={"default payment next month": "label"}, inplace=True)

print(f"Dataset loaded  : {df.shape}")
print(f"Total clients   : {len(df)}")
print(f"Defaults        : {df['label'].sum()} ({df['label'].mean()*100:.1f}%)")
print(f"Non-defaults    : {(df['label']==0).sum()} ({(df['label']==0).mean()*100:.1f}%)")

1. Evaluate clean model.

In [ ]:
# Pick 500 samples the model never saw during training
# random_state=99 makes sure we get different samples
# from the 5000 we used for training (which used random_state=42)
test_df = df.sample(n=500, random_state=99)

print(f"Test samples     : {len(test_df)}")
print(f"Defaults in test : {test_df['label'].sum()}")
print(f"Non-defaults     : {(test_df['label']==0).sum()}")

# Parse model output into 0 or 1
def text_to_label(text):
    t = text.lower().strip()
    if "no default" in t:
        return 0
    if "default" in t:
        return 1
    return 0  # if model gives unexpected output, treat as no default

# Run predictions on clean test set
print("\nEvaluating clean model...")
y_true       = []
y_pred_clean = []

for i, row in test_df.iterrows():
    prediction = predict(
        age       = int(row["AGE"]),
        sex       = int(row["SEX"]),
        marriage  = int(row["MARRIAGE"]),
        pay_0     = int(row["PAY_0"]),
        limit_bal = float(row["LIMIT_BAL"]),
    )
    y_true.append(int(row["label"]))
    y_pred_clean.append(text_to_label(prediction))

    if len(y_pred_clean) % 100 == 0:
        print(f"  ... {len(y_pred_clean)}/500 done")

# Calculate accuracy and F1 score
acc_clean = accuracy_score(y_true, y_pred_clean)
f1_clean  = f1_score(y_true, y_pred_clean, zero_division=0)

print("\n--------------------------------------")
print("   CLEAN MODEL RESULTS (BASELINE)")
print("--------------------------------------")
print(f"  Accuracy : {acc_clean:.4f} ({acc_clean*100:.1f}%)")
print(f"  F1 Score : {f1_clean:.4f}")
print("\nDetailed Report:")
print(classification_report(y_true, y_pred_clean,
                             target_names=["No default", "Default"],
                             zero_division=0))

The above cell gave 0 for Defalut, this means the model always predicting No Default for every single client. it never predicts Defalut at all. so this is called class imbalance problem.

#Balanced Dataset.    
Instead of randomly sampling 5,000 clients, we deliberately pick equal numbers of defaults and non-defaults. This way the model sees both classes equally during training and learns to predict both properly.

In [ ]:
from datasets import Dataset

def format_phi3(row):
    features = (
        f"Age: {int(row['AGE'])}, "
        f"Sex: {int(row['SEX'])}, "
        f"Marriage: {int(row['MARRIAGE'])}, "
        f"PAY_0: {int(row['PAY_0'])}, "
        f"Limit: ${int(row['LIMIT_BAL']):,}"
    )
    label = "Default" if int(row['label']) == 1 else "No default"
    return (
        f"<|user|>\n"
        f"You are a credit risk model. "
        f"Answer only with 'Default' or 'No default'.\n\n"
        f"Client:\n{features}\n"
        f"<|assistant|>\n"
        f"{label}<|end|>"
    )

# Separate defaults and non-defaults
defaults     = df[df['label'] == 1]
non_defaults = df[df['label'] == 0]

print(f"Total defaults     : {len(defaults)}")
print(f"Total non-defaults : {len(non_defaults)}")

# Sample 2,500 from each class
defaults_sample     = defaults.sample(n=2500, random_state=42)
non_defaults_sample = non_defaults.sample(n=2500, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([defaults_sample, non_defaults_sample])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset size : {len(df_balanced)}")
print(f"Defaults              : {df_balanced['label'].sum()} (50%)")
print(f"Non-defaults          : {(df_balanced['label']==0).sum()} (50%)")

# Format for Phi-3
df_balanced['text'] = df_balanced.apply(format_phi3, axis=1)
dataset = Dataset.from_pandas(df_balanced[['text']])

print(f"\nDataset ready for training : {len(dataset)} samples")
print("\nSample prompt:")
print(dataset[0]['text'])

#Reload Phi-3 fresh for retraining:

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading fresh Phi-3 base model...")
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("Model ready for retraining")

Retrain on Balanced Data:

In [ ]:
!pip install trl==0.8.6 datasets==2.21.0 -q
print("trl installed successfully")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

tokenizer.padding_side = 'right'

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/phi3-credit-balanced",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=20,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=0,
    dataloader_drop_last=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
)

print("Training starting on balanced dataset...")
print(f"Total samples  : {len(dataset)}")
print(f"Defaults       : 2500 (50%)")
print(f"Non-defaults   : 2500 (50%)")
print(f"Epochs         : 3")
print(f"Estimated time : 25-35 minutes on A100")
trainer.train()
print("Training complete!")

save the model

In [ ]:
save_dir = "/content/drive/MyDrive/phi3-credit-balanced-final"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model saved to: {save_dir}")

Reload the Balanced Model for Inference:
After training, the model is in "training mode". We need to reload it properly in "evaluation mode" so it can make clean predictions.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# New save directory for balanced model
save_dir = "/content/drive/MyDrive/phi3-credit-balanced-final"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading balanced model...")
base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = PeftModel.from_pretrained(base_model, save_dir)
model.eval()

def predict(age, sex, marriage, pay_0, limit_bal):
    prompt = (
        f"<|user|>\n"
        f"You are a credit risk model. "
        f"Answer only with 'Default' or 'No default'.\n\n"
        f"Client:\n"
        f"Age: {age}, Sex: {sex}, Marriage: {marriage}, "
        f"PAY_0: {pay_0}, Limit: ${int(limit_bal):,}\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            use_cache=True,
        )
    text   = tokenizer.decode(output[0], skip_special_tokens=True)
    answer = text.split("<|assistant|>")[-1].strip()
    return answer

print("Balanced model loaded and ready!")
print("\nQuick test:")
print("Client 1 (low risk) :", predict(44, 2, 2, 0, 50000))
print("Client 2 (high risk):", predict(24, 2, 1, 3, 7000))

In [ ]:
def predict(age, sex, marriage, pay_0, limit_bal):
    prompt = (
        f"<|user|>\n"
        f"You are a credit risk model. "
        f"Answer only with 'Default' or 'No default'.\n\n"
        f"Client:\n"
        f"Age: {age}, Sex: {sex}, Marriage: {marriage}, "
        f"PAY_0: {pay_0}, Limit: ${int(limit_bal):,}\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            use_cache=True,
        )
    # Decode only the NEW tokens generated, not the full prompt
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer

# Test again
print("Client 1 (low risk) :", predict(44, 2, 2, 0, 50000))
print("Client 2 (high risk):", predict(24, 2, 1, 3, 7000))

Evaluate the Balanced Model (Baseline):Now we test the balanced model on 500 fresh samples to get our new baseline accuracy. This time we expect to see the model predicting both Default and No default correctly — unlike before where it always said No default.

In [ ]:
test_df = df.sample(n=500, random_state=99)

print(f"Test samples     : {len(test_df)}")
print(f"Defaults in test : {test_df['label'].sum()}")
print(f"Non-defaults     : {(test_df['label']==0).sum()}")

def text_to_label(text):
    t = text.lower().strip()
    if "no default" in t:
        return 0
    if "default" in t:
        return 1
    return 0

print("\nEvaluating balanced model...")
y_true       = []
y_pred_clean = []

for i, row in test_df.iterrows():
    prediction = predict(
        age       = int(row["AGE"]),
        sex       = int(row["SEX"]),
        marriage  = int(row["MARRIAGE"]),
        pay_0     = int(row["PAY_0"]),
        limit_bal = float(row["LIMIT_BAL"]),
    )
    y_true.append(int(row["label"]))
    y_pred_clean.append(text_to_label(prediction))

    if len(y_pred_clean) % 100 == 0:
        print(f"  ... {len(y_pred_clean)}/500 done")

acc_clean = accuracy_score(y_true, y_pred_clean)
f1_clean  = f1_score(y_true, y_pred_clean, zero_division=0)

print("\n--------------------------------------")
print("   BALANCED MODEL RESULTS (BASELINE)")
print("--------------------------------------")
print(f"  Accuracy : {acc_clean:.4f} ({acc_clean*100:.1f}%)")
print(f"  F1 Score : {f1_clean:.4f}")
print("\nDetailed Report:")
print(classification_report(y_true, y_pred_clean,
                             target_names=["No default", "Default"],
                             zero_division=0))

#Label Flip Attack:






This is the actual attack. We take the same 500 test samples and flip 35% of the labels. The model stays exactly the same — only the labels change. This simulates what happens when an attacker corrupts the training data labels.

In [ ]:
FLIP_RATIO = 0.35

# Make a copy of test set to poison
poisoned_test_df = test_df.copy()

# Randomly select 35% of rows to flip
np.random.seed(42)
n_flip       = int(len(poisoned_test_df) * FLIP_RATIO)
flip_indices = np.random.choice(
    poisoned_test_df.index,
    size=n_flip,
    replace=False
)

# Flip labels: 0 → 1 and 1 → 0
poisoned_test_df.loc[flip_indices, "label"] = (
    1 - poisoned_test_df.loc[flip_indices, "label"]
)

print("--------------------------------------")
print("   LABEL FLIP ATTACK SUMMARY")
print("--------------------------------------")
print(f"  Total test samples : {len(test_df)}")
print(f"  Labels flipped     : {n_flip} ({FLIP_RATIO*100:.0f}%)")
print(f"\n  Before attack:")
print(f"    No default : {(test_df['label']==0).sum()}")
print(f"    Default    : {(test_df['label']==1).sum()}")
print(f"\n  After attack:")
print(f"    No default : {(poisoned_test_df['label']==0).sum()}")
print(f"    Default    : {(poisoned_test_df['label']==1).sum()}")

175 labels were randomly flipped
Some "No default" clients became "Default"
Some "Default" clients became "No default"
The model has no idea this happened

##Evaluate Model on Poisoned Data:
 We run the exact same model on the same 500 samples but now with flipped labels. The model's predictions haven't changed at all — only the labels changed. So when we compare predictions against flipped labels, the accuracy will drop showing the attack worked.

In [ ]:
print("Evaluating model on poisoned data...")
y_true_poisoned = []
y_pred_poisoned = []

for i, row in poisoned_test_df.iterrows():
    prediction = predict(
        age       = int(row["AGE"]),
        sex       = int(row["SEX"]),
        marriage  = int(row["MARRIAGE"]),
        pay_0     = int(row["PAY_0"]),
        limit_bal = float(row["LIMIT_BAL"]),
    )
    y_true_poisoned.append(int(row["label"]))
    y_pred_poisoned.append(text_to_label(prediction))

    if len(y_pred_poisoned) % 100 == 0:
        print(f"  ... {len(y_pred_poisoned)}/500 done")

acc_poisoned = accuracy_score(y_true_poisoned, y_pred_poisoned)
f1_poisoned  = f1_score(y_true_poisoned, y_pred_poisoned, zero_division=0)

print("\n--------------------------------------")
print("   POISONED DATA RESULTS")
print("--------------------------------------")
print(f"  Accuracy : {acc_poisoned:.4f} ({acc_poisoned*100:.1f}%)")
print(f"  F1 Score : {f1_poisoned:.4f}")
print("\nDetailed Report:")
print(classification_report(y_true_poisoned, y_pred_poisoned,
                             target_names=["No default", "Default"],
                             zero_division=0))

Final comparision

In [ ]:
acc_drop = acc_clean - acc_poisoned
f1_drop  = f1_clean  - f1_poisoned

print("══════════════════════════════════════════════")
print("   CLEAN vs POISONED - FINAL COMPARISON")
print("══════════════════════════════════════════════")
print(f"  {'Metric':<12} {'Clean':>10} {'Poisoned':>10} {'Drop':>10}")
print(f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10}")
print(f"  {'Accuracy':<12} {acc_clean:>10.4f} {acc_poisoned:>10.4f} {acc_drop:>+10.4f}")
print(f"  {'F1 Score':<12} {f1_clean:>10.4f} {f1_poisoned:>10.4f} {f1_drop:>+10.4f}")
print("══════════════════════════════════════════════")
print(f"\n  Attack configuration:")
print(f"  Flip ratio     : {FLIP_RATIO*100:.0f}%")
print(f"  Labels flipped : {n_flip} / {len(test_df)}")
print(f"  Test samples   : {len(test_df)}")
print(f"\n  Attack result:")
print(f"  Accuracy drop  : {acc_drop*100:.1f}%")
print(f"  F1 drop        : {f1_drop:.4f}")
print(f"  Attack success : {'YES' if acc_drop > 0.05 else 'MINIMAL'}")

In real world terms:
This means if a bank was using this model and an attacker flipped 35% of the training labels:

Risky clients would be approved for credit they shouldn't get
Safe clients would be incorrectly flagged as risky
The bank would lose money and make wrong decisions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Graph 1: Accuracy and F1 Comparison Bar Chart ────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Label Flip Poisoning Attack - Results',
             fontsize=14, fontweight='bold')

# Bar chart - Accuracy and F1
metrics     = ['Accuracy', 'F1 Score']
clean_scores   = [acc_clean, f1_clean]
poisoned_scores = [acc_poisoned, f1_poisoned]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, clean_scores, width,
            label='Clean', color='steelblue')
axes[0].bar(x + width/2, poisoned_scores, width,
            label='Poisoned', color='tomato')
axes[0].set_title('Clean vs Poisoned Model Performance')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Score')
axes[0].legend()

# Add value labels on bars
for i, (c, p) in enumerate(zip(clean_scores, poisoned_scores)):
    axes[0].text(i - width/2, c + 0.01, f'{c:.2f}',
                ha='center', fontsize=10)
    axes[0].text(i + width/2, p + 0.01, f'{p:.2f}',
                ha='center', fontsize=10)

# ── Graph 2: Label Distribution Before and After Attack ─
categories = ['No Default', 'Default']
before = [(test_df['label']==0).sum(),
          (test_df['label']==1).sum()]
after  = [(poisoned_test_df['label']==0).sum(),
          (poisoned_test_df['label']==1).sum()]

x = np.arange(len(categories))
axes[1].bar(x - width/2, before, width,
            label='Before Attack', color='steelblue')
axes[1].bar(x + width/2, after, width,
            label='After Attack', color='tomato')
axes[1].set_title('Label Distribution Before vs After Attack')
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].set_ylabel('Number of Samples')
axes[1].legend()

# Add value labels
for i, (b, a) in enumerate(zip(before, after)):
    axes[1].text(i - width/2, b + 2, str(b),
                ha='center', fontsize=10)
    axes[1].text(i + width/2, a + 2, str(a),
                ha='center', fontsize=10)

# ── Graph 3: Per Class Performance ───────────────────
classes    = ['No Default', 'Default']
clean_recall   = [0.87, 0.58]
poisoned_recall = [0.79, 0.25]

x = np.arange(len(classes))
axes[2].bar(x - width/2, clean_recall, width,
            label='Clean', color='steelblue')
axes[2].bar(x + width/2, poisoned_recall, width,
            label='Poisoned', color='tomato')
axes[2].set_title('Recall Per Class Before vs After Attack')
axes[2].set_xticks(x)
axes[2].set_xticklabels(classes)
axes[2].set_ylim(0, 1)
axes[2].set_ylabel('Recall Score')
axes[2].legend()

# Add value labels
for i, (c, p) in enumerate(zip(clean_recall, poisoned_recall)):
    axes[2].text(i - width/2, c + 0.01, f'{c:.2f}',
                ha='center', fontsize=10)
    axes[2].text(i + width/2, p + 0.01, f'{p:.2f}',
                ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/label_flip_results.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Graph saved to Google Drive")

In [ ]:
# quick check
print("df shape        :", df.shape)
print("test_df shape   :", test_df.shape)
print("acc_clean       :", acc_clean)
print("acc_poisoned    :", acc_poisoned)

#DEFENSES

##ISOLATION FOREST DEFENSE:

isolation Forest looks at the features of each training sample and finds ones that look suspicious or unusual. When an attacker flips labels, the features and labels no longer match — Isolation Forest can detect these mismatches and remove them before retraining.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# ── Step 1: Prepare the poisoned training data ────────
# We simulate the attacker poisoned our training data
# by flipping 35% of labels in df_balanced

poisoned_train_df = df_balanced.copy()

np.random.seed(42)
n_flip_train   = int(len(poisoned_train_df) * 0.35)
flip_idx_train = np.random.choice(
    poisoned_train_df.index,
    size=n_flip_train,
    replace=False
)
poisoned_train_df.loc[flip_idx_train, "label"] = (
    1 - poisoned_train_df.loc[flip_idx_train, "label"]
)

print("Poisoned training data created")
print(f"Total samples    : {len(poisoned_train_df)}")
print(f"Labels flipped   : {n_flip_train}")
print(f"Defaults         : {poisoned_train_df['label'].sum()}")
print(f"Non-defaults     : {(poisoned_train_df['label']==0).sum()}")

# ── Step 2: Select features for Isolation Forest ──────
features = ['AGE', 'SEX', 'MARRIAGE', 'PAY_0', 'LIMIT_BAL']

X_train = poisoned_train_df[features].values
y_train = poisoned_train_df['label'].values

# ── Step 3: Scale features ────────────────────────────
# Isolation Forest works better when all features
# are on the same scale
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

# ── Step 4: Train Isolation Forest ───────────────────
# contamination=0.35 tells it to expect 35% anomalies
# which matches our flip ratio
print("\nTraining Isolation Forest...")
iso_forest = IsolationForest(
    contamination=0.35,
    random_state=42,
    n_estimators=100
)
iso_forest.fit(X_scaled)

# ── Step 5: Detect anomalies ──────────────────────────
# Returns 1 for normal samples, -1 for anomalies
predictions_if = iso_forest.predict(X_scaled)
normal_mask    = predictions_if == 1

print(f"Total samples    : {len(X_train)}")
print(f"Normal samples   : {normal_mask.sum()}")
print(f"Anomalies removed: {(~normal_mask).sum()}")

# ── Step 6: Keep only normal samples ─────────────────
clean_train_df = poisoned_train_df[normal_mask].reset_index(drop=True)

print(f"\nCleaned training data:")
print(f"Samples remaining : {len(clean_train_df)}")
print(f"Defaults          : {clean_train_df['label'].sum()}")
print(f"Non-defaults      : {(clean_train_df['label']==0).sum()}")

Retrain on Cleaned Data:


In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ── Step 1: Format cleaned data for Phi-3 ────────────
clean_train_df['text'] = clean_train_df.apply(
    format_phi3, axis=1
)
clean_dataset = Dataset.from_pandas(clean_train_df[['text']])

print(f"Clean dataset size : {len(clean_dataset)}")
print(f"Sample prompt:")
print(clean_dataset[0]['text'])

# ── Step 2: Load fresh Phi-3 ─────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("\nLoading fresh Phi-3 for retraining...")
defended_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

tokenizer2 = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct"
)
tokenizer2.pad_token    = tokenizer2.eos_token
tokenizer2.padding_side = "right"

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

defended_model = get_peft_model(defended_model, peft_config)
defended_model.print_trainable_parameters()

# ── Step 3: Train on cleaned data ────────────────────
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/phi3-credit-defended",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=20,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=0,
    dataloader_drop_last=True,
)

trainer = SFTTrainer(
    model=defended_model,
    train_dataset=clean_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer2,
    args=training_args,
    packing=False,
)

print("\nRetraining on cleaned data...")
print(f"Samples : {len(clean_dataset)}")
print(f"Epochs  : 3")
print(f"Estimated time : 25-30 minutes")
trainer.train()
print("Retraining complete!")

In [ ]:
import os

save_dir_defended = "/content/drive/MyDrive/phi3-credit-isolation-final"
trainer.model.save_pretrained(save_dir_defended)
tokenizer2.save_pretrained(save_dir_defended)
print(f"Defended model saved to: {save_dir_defended}")

Reload Defended Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

save_dir_defended = "/content/drive/MyDrive/phi3-credit-isolation-final"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading defended model...")
base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct"
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

defended_model = PeftModel.from_pretrained(base_model, save_dir_defended)
defended_model.eval()

def predict_defended(age, sex, marriage, pay_0, limit_bal):
    prompt = (
        f"<|user|>\n"
        f"You are a credit risk model. "
        f"Answer only with 'Default' or 'No default'.\n\n"
        f"Client:\n"
        f"Age: {age}, Sex: {sex}, Marriage: {marriage}, "
        f"PAY_0: {pay_0}, Limit: ${int(limit_bal):,}\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(defended_model.device)
    with torch.no_grad():
        output = defended_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            use_cache=True,
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer

print("Defended model loaded!")
print("\nQuick test:")
print("Client 1 (low risk) :", predict_defended(44, 2, 2, 0, 50000))
print("Client 2 (high risk):", predict_defended(24, 2, 1, 3, 7000))

evaluate defended model.

In [ ]:
print("Evaluating defended model on poisoned test set...")
y_pred_defended = []

for i, row in poisoned_test_df.iterrows():
    prediction = predict_defended(
        age       = int(row["AGE"]),
        sex       = int(row["SEX"]),
        marriage  = int(row["MARRIAGE"]),
        pay_0     = int(row["PAY_0"]),
        limit_bal = float(row["LIMIT_BAL"]),
    )
    y_pred_defended.append(text_to_label(prediction))

    if len(y_pred_defended) % 100 == 0:
        print(f"  ... {len(y_pred_defended)}/500 done")

acc_defended = accuracy_score(y_true_poisoned, y_pred_defended)
f1_defended  = f1_score(y_true_poisoned, y_pred_defended, zero_division=0)

print("\n--------------------------------------")
print("   DEFENDED MODEL RESULTS")
print("--------------------------------------")
print(f"  Accuracy : {acc_defended:.4f} ({acc_defended*100:.1f}%)")
print(f"  F1 Score : {f1_defended:.4f}")
print("\nDetailed Report:")
print(classification_report(y_true_poisoned, y_pred_defended,
                             target_names=["No default", "Default"],
                             zero_division=0))

In [ ]:
print("Evaluating defended model on CLEAN test set...")
y_pred_defended_clean = []

for i, row in test_df.iterrows():
    prediction = predict_defended(
        age       = int(row["AGE"]),
        sex       = int(row["SEX"]),
        marriage  = int(row["MARRIAGE"]),
        pay_0     = int(row["PAY_0"]),
        limit_bal = float(row["LIMIT_BAL"]),
    )
    y_pred_defended_clean.append(text_to_label(prediction))

    if len(y_pred_defended_clean) % 100 == 0:
        print(f"  ... {len(y_pred_defended_clean)}/500 done")

acc_defended_clean = accuracy_score(y_true, y_pred_defended_clean)
f1_defended_clean  = f1_score(y_true, y_pred_defended_clean, zero_division=0)

print("\n--------------------------------------")
print("   DEFENDED MODEL ON CLEAN TEST SET")
print("--------------------------------------")
print(f"  Accuracy : {acc_defended_clean:.4f} ({acc_defended_clean*100:.1f}%)")
print(f"  F1 Score : {f1_defended_clean:.4f}")
print("\nDetailed Report:")
print(classification_report(y_true, y_pred_defended_clean,
                             target_names=["No default", "Default"],
                             zero_division=0))

full comparision

In [ ]:
print("══════════════════════════════════════════════════════")
print("   COMPLETE RESULTS - LABEL FLIP ATTACK AND DEFENSE")
print("══════════════════════════════════════════════════════")
print(f"  {'Model':<30} {'Accuracy':>10} {'F1 Score':>10}")
print(f"  {'-'*30} {'-'*10} {'-'*10}")
print(f"  {'Clean Model (baseline)':<30} {acc_clean:>10.4f} {f1_clean:>10.4f}")
print(f"  {'After Label Flip Attack':<30} {acc_poisoned:>10.4f} {f1_poisoned:>10.4f}")
print(f"  {'After Isolation Forest':<30} {acc_defended_clean:>10.4f} {f1_defended_clean:>10.4f}")
print("══════════════════════════════════════════════════════")
print(f"\n  Attack impact:")
print(f"  Accuracy drop    : {(acc_clean - acc_poisoned)*100:.1f}%")
print(f"  F1 drop          : {f1_clean - f1_poisoned:.4f}")
print(f"\n  Defense recovery:")
print(f"  Accuracy gained  : {(acc_defended_clean - acc_poisoned)*100:.1f}%")
print(f"  F1 gained        : {f1_defended_clean - f1_poisoned:.4f}")
print(f"\n  Remaining gap from clean baseline:")
print(f"  Accuracy gap     : {(acc_clean - acc_defended_clean)*100:.1f}%")
print(f"  F1 gap           : {f1_clean - f1_defended_clean:.4f}")

final visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Label Flip Attack and Isolation Forest Defense Results',
             fontsize=13, fontweight='bold')

# ── Graph 1: Accuracy comparison ─────────────────────
models    = ['Clean\nModel', 'After\nAttack', 'After\nDefense']
accuracies = [acc_clean, acc_poisoned, acc_defended_clean]
colors     = ['steelblue', 'tomato', 'green']

bars = axes[0].bar(models, accuracies, color=colors, width=0.5)
axes[0].set_title('Accuracy Comparison')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1)

for bar, val in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                val + 0.01, f'{val*100:.1f}%',
                ha='center', fontsize=11, fontweight='bold')

# ── Graph 2: F1 Score comparison ──────────────────────
f1_scores = [f1_clean, f1_poisoned, f1_defended_clean]

bars = axes[1].bar(models, f1_scores, color=colors, width=0.5)
axes[1].set_title('F1 Score Comparison')
axes[1].set_ylabel('F1 Score')
axes[1].set_ylim(0, 1)

for bar, val in zip(bars, f1_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                val + 0.01, f'{val:.3f}',
                ha='center', fontsize=11, fontweight='bold')

# ── Graph 3: Attack and Defense impact ────────────────
categories  = ['Accuracy Drop\n(Attack)',
               'Accuracy Recovered\n(Defense)',
               'Remaining Gap']
values      = [
    (acc_clean - acc_poisoned) * 100,
    (acc_defended_clean - acc_poisoned) * 100,
    (acc_clean - acc_defended_clean) * 100
]
bar_colors  = ['tomato', 'green', 'orange']

bars = axes[2].bar(categories, values, color=bar_colors, width=0.5)
axes[2].set_title('Attack vs Defense Impact')
axes[2].set_ylabel('Percentage (%)')

for bar, val in zip(bars, values):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                val + 0.2, f'{val:.1f}%',
                ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/attack_defense_results.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Graph saved to Google Drive")